In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path

# src 경로 등록 + config 로드
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from bootstrap import setup_notebook
from unet import get_target_block

ROOT, cfg, _vars = setup_notebook("apple.yaml") # target concept
globals().update(_vars)

import os
import json
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import torch
from diffusers import StableDiffusionPipeline

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [2]:
if "pipe" not in globals():
    pipe = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5")
    pipe = pipe.to(DEVICE)
    pipe.safety_checker = None
    pipe.set_progress_bar_config(disable=True)


`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["id2label"]` will be overriden.
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["bos_token_id"]` will be overriden.
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["eos_token_id"]` will be overriden.


In [3]:
images, ff_in_list = [], []
prompt_index = []

for p_idx, prompt in enumerate(PROMPTS):
    pdir = resolve_pdir(p_idx, prompt)
    n    = NUM_IMAGES_PER_PROMPT
    images.extend([Image.open(pdir / "images" / f"{i:04d}.png") for i in range(n)])
    act = torch.load(pdir / TARGET_BLOCK.replace(".", "_") / "activations.pt", weights_only=True)
    try:
        ff_in_list.append(act["inputs"][:n].cpu().float())
    except Exception:
        ff_in_list.append(act["ff_in"][:n].cpu().float())
    prompt_index.extend([p_idx] * n)

ff_in        = torch.cat(ff_in_list, dim=0)
prompt_index = np.array(prompt_index, dtype=np.int32)
N            = len(images)

N, seq_len, dim = ff_in.shape
side = int(seq_len ** 0.5)
print(f"ff_in: {tuple(ff_in.shape)}  side={side}")


ff_in: (100, 256, 1280)  side=16


\begin{aligned}
&\text{직선의 방정식} \\
&w_1 x + w_2 y + b_i = 0 \\
&⇒ w_2 y = -(w_1 x + b_i) \\
&⇒ y = -\frac{w_1 x + b_i}{w_2}
\end{aligned}

In [4]:
from hyperplane import extract_w2_b2

W2, b2 = extract_w2_b2(pipe, TARGET_BLOCK)
print(f"W2: {tuple(W2.shape)}  b2: {tuple(b2.shape)}")

W2: (5120, 1280)  b2: (5120,)


In [5]:
# ff_in: (N, 256, 1280) — 이미지마다 mean_map·similarity·top-k를 각각 계산
N, seq_len, dim = ff_in.shape
side = int(seq_len ** 0.5)
assert side * side == seq_len

K = 100 # 추출한 주요 뉴런 수

# (N, side, side, dim) → spatial mean → (N, seq_len)
h_3d = ff_in.reshape(N, side, side, dim)
mean_map = h_3d.mean(dim=-1).reshape(N, seq_len)  # (N, 256)

# h: (N, 256, 1280) @ W2.T: (1280, 5120) → (N, 256, 5120)
neuron_acts = ff_in @ W2.T

# spatial mean과의 유사도
similarity = (neuron_acts * mean_map.unsqueeze(-1)).sum(dim=1)  # (N, 5120)

similarity_mean = similarity.mean(dim=0)  # (5120,)
top_k_idx = similarity_mean.topk(K).indices  # 0~5119 범위

In [6]:
# w & b도 해당 뉴런들만 추출
W2_topk = W2#[top_k_idx, :]
b2_topk = b2#[top_k_idx]

print(W2_topk.shape, b2_topk.shape)

torch.Size([5120, 1280]) torch.Size([5120])


In [7]:
# 각 토큰이 hyperplane에서 얼마나 떨어져 있는지
signed_dist = ff_in @ W2_topk.T
signed_dist = signed_dist + b2_topk

signed_dist.shape

torch.Size([100, 256, 5120])

In [8]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# 1. PCA
h_all = ff_in.reshape(-1, ff_in.shape[-1]).cpu().numpy()
pca_act = PCA(n_components=2)
h_2d = pca_act.fit_transform(h_all)

# 범주
x_min, x_max = h_2d[:, 0].min(), h_2d[:, 0].max()
y_min, y_max = h_2d[:, 1].min(), h_2d[:, 1].max()
pad = 0.05
x_lim = (x_min - pad*(x_max-x_min), x_max + pad*(x_max-x_min))
y_lim = (y_min - pad*(y_max-y_min), y_max + pad*(y_max-y_min))
x_range = np.linspace(x_lim[0], x_lim[1], 300)

# 2. w_j, b_j를 PCA 공간으로 변환
def to_2d(w_j, b_j):
    w_2d = pca_act.components_ @ w_j.cpu().numpy()
    b_2d = float(w_j.cpu().numpy() @ pca_act.mean_) + float(b_j)
    return w_2d, b_2d

# 3. 교점 계산
intersections = []
for i in range(K):
    for j in range(i+1, K):
        w_2d_i, b_2d_i = to_2d(W2_topk[i], b2_topk[i])
        w_2d_j, b_2d_j = to_2d(W2_topk[j], b2_topk[j])

        A = np.array([w_2d_i, w_2d_j])
        b = np.array([-b_2d_i, -b_2d_j])

        if abs(np.linalg.det(A)) < 1e-8:
            continue

        pt = np.linalg.solve(A, b)
        if x_lim[0] < pt[0] < x_lim[1] and y_lim[0] < pt[1] < y_lim[1]:
            intersections.append(pt)

intersections = np.array(intersections) if intersections else np.zeros((0, 2))

# # 4. 시각화 — 뉴런 k마다
# n_pts = h_2d.shape[0]
# sd_flat = signed_dist.reshape(n_pts, -1).cpu().numpy()

# w_2d_all, b_2d_all = [], []
# for j in range(K):
#     w, b = to_2d(W2_topk[j], b2_topk[j])
#     w_2d_all.append(w)
#     b_2d_all.append(b)

# ncols = 10
# nrows = int(np.ceil(K / ncols))
# fig, axes = plt.subplots(nrows, ncols, figsize=(2.2 * ncols, 2.2 * nrows), squeeze=False)
# axes_flat = axes.flatten()

# def _plot_boundary(ax, w_2d, b_2d, *, lw=0.5, alpha=0.3, color='k'):
#     if abs(w_2d[1]) < 1e-8:
#         if abs(w_2d[0]) > 1e-8:
#             x0 = -b_2d / w_2d[0]
#             if x_lim[0] <= x0 <= x_lim[1]:
#                 ax.axvline(x0, color=color, lw=lw, alpha=alpha)
#         return
#     y_line = -(w_2d[0] * x_range + b_2d) / w_2d[1]
#     mask = (y_line >= y_lim[0]) & (y_line <= y_lim[1])
#     if mask.any():
#         ax.plot(x_range[mask], y_line[mask], '-', color=color, lw=lw, alpha=alpha)

# for k, ax in enumerate(axes_flat[:K]):
#     sc = ax.scatter(
#         h_2d[:, 0], h_2d[:, 1],
#         c=sd_flat[:, k], cmap='RdBu', s=2, alpha=0.35, vmin=-3, vmax=3,
#     )
#     for j in range(K):
#         if j == k:
#             _plot_boundary(ax, w_2d_all[j], b_2d_all[j], lw=1.2, alpha=0.9, color='crimson')
#         else:
#             _plot_boundary(ax, w_2d_all[j], b_2d_all[j], lw=0.4, alpha=0.25, color='k')
#     ax.set_xlim(x_lim)
#     ax.set_ylim(y_lim)
#     ax.axis('off')
#     ax.set_title(f'neuron {k}', fontsize=8)
#     ax.tick_params(labelsize=6)

# for ax in axes_flat[K:]:
#     ax.axis('off')

# plt.show()

print('PCA explained variance:', pca_act.explained_variance_ratio_)
print('교점 수:', len(intersections))

PCA explained variance: [0.15610722 0.07296225]
교점 수: 622


In [9]:
# downstream 노트북용 캐시 저장
from load import save_cache
save_cache(
    cfg, "load_and_analyze",
    ff_in=ff_in, W2=W2, b2=b2,
    top_k_idx=top_k_idx, signed_dist=signed_dist,
    W2_topk=W2_topk, b2_topk=b2_topk, N=torch.tensor(N),
)


cache saved → /project/dam/data/apple/up_blocks_1/_cache/load_and_analyze.pt


PosixPath('/project/dam/data/apple/up_blocks_1/_cache/load_and_analyze.pt')